# Revenue Dashboard



In [1]:
!pip install dash


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import plotly.express as px
from pathlib import Path
import dash
from dash import dcc, html, Input, Output
import webbrowser

# =========================================================
# LOAD DATA
# =========================================================
base = Path("../data/gold")

# Main fact table (the one you used to build all others)
df = pd.read_csv(base / "latest_clean_dataset.csv")

# ---------------------------------------------------------
# BASIC CLEANUP / DERIVED COLUMNS
# ---------------------------------------------------------
# Strip column names
df.columns = df.columns.str.strip()

# Make sure key columns exist with consistent names
# Adjust these if your actual column names differ
if "InvoiceDate" in df.columns:
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
    df["Year"] = df["InvoiceDate"].dt.year
    df["Month"] = df["InvoiceDate"].dt.month
    df["Week"] = df["InvoiceDate"].dt.isocalendar().week

# Ensure numeric
for col in ["TotalPrice", "Quantity", "OrderRevenue"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Decide which revenue column to use
revenue_col = "OrderRevenue" if "OrderRevenue" in df.columns else "TotalPrice"

# =========================================================
# FILTER OPTIONS (from main table)
# =========================================================
country_col = "Country"
category_col = "ProductCategory"

country_options = [
    {"label": c, "value": c}
    for c in sorted(df[country_col].dropna().unique())
]

category_options = [
    {"label": c, "value": c}
    for c in sorted(df[category_col].dropna().unique())
]

# =========================================================
# STYLES
# =========================================================
page_style = {
    "backgroundColor": "#0B1020",
    "minHeight": "100vh",
    "paddingBottom": "30px",
    "fontFamily": "Segoe UI"
}

card_style = {
    "backgroundColor": "#131C31",
    "borderRadius": "18px",
    "padding": "15px",
    "boxShadow": "0px 6px 18px rgba(0,0,0,0.35)"
}

chart_style = {
    "backgroundColor": "#131C31",
    "borderRadius": "18px",
    "padding": "15px",
    "boxShadow": "0px 6px 18px rgba(0,0,0,0.35)"
}

filter_label = {
    "color": "white",
    "fontWeight": "600",
    "marginBottom": "5px"
}

# KPI styles
kpi_common = {
    "padding": "20px",
    "borderRadius": "18px",
    "color": "white",
    "textAlign": "center",
    "boxShadow": "0px 6px 18px rgba(0,0,0,0.3)"
}

kpi1 = {**kpi_common, "background": "linear-gradient(135deg,#4facfe,#00f2fe)"}
kpi2 = {**kpi_common, "background": "linear-gradient(135deg,#43e97b,#38f9d7)"}
kpi3 = {**kpi_common, "background": "linear-gradient(135deg,#fa709a,#fee140)"}
kpi4 = {**kpi_common, "background": "linear-gradient(135deg,#667eea,#764ba2)"}

# =========================================================
# DASH APP
# =========================================================
app = dash.Dash(__name__)
server = app.server

# =========================================================
# LAYOUT
# =========================================================
app.layout = html.Div(style=page_style, children=[

    # HEADER
    html.Div([
        html.H1(
            "📊 Executive Revenue Dashboard",
            style={
                "textAlign": "center",
                "color": "white",
                "padding": "20px",
                "marginBottom": "10px"
            }
        )
    ]),

    # FILTER SECTION
    html.Div(style={
        "width": "92%",
        "margin": "auto"
    }, children=[

        html.Div(style=card_style, children=[

            html.Div(style={
                "display": "grid",
                "gridTemplateColumns": "1fr 1fr 1fr 1fr",
                "gap": "15px"
            }, children=[

                # COUNTRY
                html.Div([
                    html.Label("🌍 Country Filter", style=filter_label),
                    dcc.Dropdown(
                        id="country",
                        options=country_options,
                        multi=True,
                        placeholder="Select Country"
                    )
                ]),

                # CATEGORY
                html.Div([
                    html.Label("📦 Product Category", style=filter_label),
                    dcc.Dropdown(
                        id="category",
                        options=category_options,
                        multi=True,
                        placeholder="Select Category"
                    )
                ]),

                # MONTH / WEEK
                html.Div([
                    html.Label("📅 Revenue Trend Type", style=filter_label),
                    dcc.Dropdown(
                        id="time_type",
                        options=[
                            {"label": "Monthly Revenue", "value": "month"},
                            {"label": "Weekly Revenue", "value": "week"}
                        ],
                        value="month",
                        clearable=False
                    )
                ]),

                # TOP N
                html.Div([
                    html.Label("🏆 Top Products Filter", style=filter_label),
                    dcc.Dropdown(
                        id="top_n",
                        options=[
                            {"label": "Top 5 Products", "value": 5},
                            {"label": "Top 10 Products", "value": 10},
                            {"label": "Top 20 Products", "value": 20},
                        ],
                        value=10,
                        clearable=False
                    )
                ]),

            ])
        ])
    ]),

    # KPI ROW
    html.Div(style={
        "display": "grid",
        "gridTemplateColumns": "repeat(4, 1fr)",
        "gap": "18px",
        "width": "92%",
        "margin": "25px auto"
    }, children=[

        html.Div(style=kpi1, children=[
            html.H4("💰 Total Revenue"),
            html.H2(id="kpi_total_revenue")
        ]),

        html.Div(style=kpi2, children=[
            html.H4("📦 Products"),
            html.H2(id="kpi_products")
        ]),

        html.Div(style=kpi3, children=[
            html.H4("🛒 Quantity Sold"),
            html.H2(id="kpi_quantity")
        ]),

        html.Div(style=kpi4, children=[
            html.H4("🌎 Countries"),
            html.H2(id="kpi_countries")
        ]),
    ]),

    # CHART GRID
    html.Div(style={
        "display": "grid",
        "gridTemplateColumns": "1fr 1fr",
        "gap": "20px",
        "width": "92%",
        "margin": "auto"
    }, children=[

        # MAP
        html.Div(style=chart_style, children=[
            html.H3("🌍 Revenue by Country", style={"color": "white"}),
            dcc.Graph(id="map", style={"height": "430px"})
        ]),

        # CATEGORY PIE
        html.Div(style=chart_style, children=[
            html.H3("📦 Revenue by Product Category", style={"color": "white"}),
            dcc.Graph(id="product", style={"height": "430px"})
        ]),

        # TOP PRODUCTS
        html.Div(style=chart_style, children=[
            html.H3("🏆 Top Products (Revenue vs Quantity)", style={"color": "white"}),
            dcc.Graph(id="top_products", style={"height": "430px"})
        ]),

        # TREND
        html.Div(style=chart_style, children=[
            html.H3("📈 Revenue Trend", style={"color": "white"}),
            dcc.Graph(id="trend_chart", style={"height": "430px"})
        ]),
    ])
])

# =========================================================
# CALLBACK
# =========================================================
@app.callback(
    Output("kpi_total_revenue", "children"),
    Output("kpi_products", "children"),
    Output("kpi_quantity", "children"),
    Output("kpi_countries", "children"),
    Output("map", "figure"),
    Output("product", "figure"),
    Output("top_products", "figure"),
    Output("trend_chart", "figure"),
    Input("country", "value"),
    Input("category", "value"),
    Input("time_type", "value"),
    Input("top_n", "value"),
)
def update_dashboard(countries, categories, time_type, top_n):

    # -----------------------------------------------------
    # APPLY FILTERS ON MAIN TABLE
    # -----------------------------------------------------
    dff = df.copy()

    if countries:
        dff = dff[dff[country_col].isin(countries)]

    if categories:
        dff = dff[dff[category_col].isin(categories)]

    # -----------------------------------------------------
    # KPIs (from filtered data)
    # -----------------------------------------------------
    total_revenue = dff[revenue_col].sum()
    total_products = dff["StockCode"].nunique() if "StockCode" in dff.columns else 0
    total_quantity = dff["Quantity"].sum() if "Quantity" in dff.columns else 0
    total_countries = dff[country_col].nunique()

    kpi_total_revenue = f"${total_revenue:,.0f}"
    kpi_products = f"{total_products:,}"
    kpi_quantity = f"{total_quantity:,.0f}"
    kpi_countries = f"{total_countries:,}"

    # -----------------------------------------------------
    # COUNTRY REVENUE (MAP)
    # -----------------------------------------------------
    df_map = (
        dff.groupby(country_col, as_index=False)[revenue_col]
        .sum()
        .rename(columns={revenue_col: "TotalRevenue"})
    )

    fig_map = px.choropleth(
        df_map,
        locations=country_col,
        locationmode="country names",
        color="TotalRevenue",
        color_continuous_scale=[
            "#f1a6f1",
            "#d6fa52",
            "#1BC9C0",
            "#73ec10"
        ]
    )

    fig_map.update_geos(
        showframe=False,
        showcoastlines=False,
        showcountries=False,
        projection_type="natural earth",
        bgcolor="rgba(0,0,0,0)",
        showland=True,
        landcolor="#EAF2FF"
    )

    fig_map.update_layout(
        paper_bgcolor="#131C31",
        plot_bgcolor="#131C31",
        margin=dict(l=0, r=0, t=0, b=0),
        font_color="white"
    )

    # -----------------------------------------------------
    # PRODUCT CATEGORY PIE
    # -----------------------------------------------------
    df_cat = (
        dff.groupby(category_col, as_index=False)[revenue_col]
        .sum()
        .rename(columns={revenue_col: "TotalRevenue"})
    )

    fig_product = px.pie(
        df_cat,
        names=category_col,
        values="TotalRevenue",
        hole=0.4
    )

    fig_product.update_layout(
        paper_bgcolor="#131C31",
        font_color="white"
    )

    # -----------------------------------------------------
    # TOP PRODUCTS (BAR)
    # -----------------------------------------------------
    group_cols = ["StockCode", "Description", category_col]
    for col in group_cols:
        if col not in dff.columns:
            # If something is missing, create a safe placeholder
            dff[col] = dff.get(col, "Unknown")

    df_top = (
        dff.groupby(group_cols, as_index=False)
        .agg(
            TotalRevenue=(revenue_col, "sum"),
            TotalQuantity=("Quantity", "sum")
        )
        .sort_values("TotalRevenue", ascending=False)
        .head(top_n)
    )

    fig_top = px.bar(
        df_top,
        x="Description",
        y=["TotalRevenue", "TotalQuantity"],
        barmode="group"
    )

    fig_top.update_layout(
        paper_bgcolor="#131C31",
        plot_bgcolor="#131C31",
        font_color="white",
        xaxis_tickangle=-45
    )

    # -----------------------------------------------------
    # MONTHLY / WEEKLY TREND
    # -----------------------------------------------------
    if time_type == "week":
        dfw = (
            dff.groupby(["Year", "Week"], as_index=False)[revenue_col]
            .sum()
            .rename(columns={revenue_col: "TotalRevenue"})
        )

        fig_trend = px.line(
            dfw,
            x="Week",
            y="TotalRevenue",
            markers=True,
            title="Weekly Revenue Trend"
        )

    else:
        dfm = (
            dff.groupby(["Year", "Month"], as_index=False)[revenue_col]
            .sum()
            .rename(columns={revenue_col: "TotalRevenue"})
        )

        # Create MonthName for nicer x-axis
        dfm["Date"] = pd.to_datetime(
            dfm["Year"].astype(str) + "-" + dfm["Month"].astype(str) + "-01",
            errors="coerce"
        )
        dfm = dfm.dropna(subset=["Date"]).sort_values("Date")
        dfm["MonthName"] = dfm["Date"].dt.strftime("%b")

        fig_trend = px.line(
            dfm,
            x="MonthName",
            y="TotalRevenue",
            markers=True,
            title="Monthly Revenue Trend"
        )

    fig_trend.update_layout(
        paper_bgcolor="#131C31",
        plot_bgcolor="#131C31",
        font_color="white"
    )

    return (
        kpi_total_revenue,
        kpi_products,
        kpi_quantity,
        kpi_countries,
        fig_map,
        fig_product,
        fig_top,
        fig_trend,
    )

# =========================================================
# RUN APP
# =========================================================
if __name__ == "__main__":
    webbrowser.open("http://127.0.0.1:8080")
    app.run(host="127.0.0.1", port=8080, debug=False)
